In [1]:
import random
import pandas as pd

In [5]:
data = pd.read_csv('shit.csv', encoding='latin1')
data.head()

,disease,symptoms
0,Common Cold,"runny nose,sneezing,sore throat,cough,congesti..."
1,Influenza,"fever,chills,cough,fatigue,muscle pain,headach..."
2,COVID-19,"fever,dry cough,fatigue,loss of smell,loss of ..."
3,Pneumonia,"fever,cough,chest pain,shortness of breath,chi..."
4,Bronchitis,"cough,mucus production,fatigue,shortness of br..."


In [33]:
a = ['a','b','c']
new_a = random.sample(a, 2)
if a==new_a:
    print(f'fuck this shit {new_a}')
else:
    print(new_a)

['c', 'b']


In [26]:
type(data.symptoms)

pandas.core.series.Series

In [ ]:
for _ in data.symptoms:
    disease_lst = _.split(',')
    length = len(disease_lst)

In [37]:
from itertools import combinations
for _ in data.symptoms:
    disease_lst = _.split(',')
    combos = list(combinations(disease_lst,3))

In [38]:
print(combos)

[('skin redness', 'swelling', 'pain')]


In [40]:
import pandas as pd
from itertools import combinations

# load dataset
data = pd.read_csv('shit.csv', encoding='latin1')

dataset = []

for index, row in data.iterrows():
    
    disease = row['disease']
    
    # convert symptom string into list
    symptoms = [s.strip() for s in row['symptoms'].split(',')]
    
    # generate combinations of 3,4,5 symptoms
    for r in range(3,6):
        
        for combo in combinations(symptoms, r):
            
            symptom_string = ", ".join(combo)
            
            dataset.append([disease, symptom_string])

# convert to dataframe
df = pd.DataFrame(dataset, columns=["disease","symptoms"])

# shuffle rows (important for ML training)
df = df.sample(frac=1).reset_index(drop=True)

# save dataset
df.to_csv("curewise_dataset.csv", index=False)

print("Dataset created:", df.shape)

Dataset created: (1086, 2)


In [41]:
data = pd.read_csv("curewise_dataset.csv")

balanced = (
    data.groupby("disease")
    .apply(lambda x: x.sample(min(len(x), 42)))
    .reset_index(drop=True)
)

balanced.to_csv("curewise_balanced.csv", index=False)

C:\Users\gauta\AppData\Local\Temp\ipykernel_3864\3471001990.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 42)))


In [42]:
data = pd.read_csv("curewise_balanced.csv")
data.head()

,disease,symptoms
0,Acne,"pimples, redness, skin inflammation, whiteheads"
1,Acne,"pimples, redness, blackheads"
2,Acne,"pimples, whiteheads, blackheads"
3,Acne,"pimples, redness, whiteheads"
4,Acne,"pimples, redness, skin inflammation"


In [47]:
data = data.sample(frac=1).reset_index(drop=True)

In [48]:
import pandas as pd

data = pd.read_csv("curewise_balanced.csv")

# shuffle rows
data = data.sample(frac=1).reset_index(drop=True)

# save shuffled file
data.to_csv("curewise_dataset_shuffled.csv", index=False)

In [49]:
import numpy as np
import pandas as pd

In [50]:
data = pd.read_csv("curewise_dataset_shuffled.csv", encoding='latin1')
data.head()

,disease,symptoms
0,Arthritis,"joint pain, stiffness, swelling"
1,Appendicitis,"severe abdominal pain, vomiting, fever, loss o..."
2,Hepatitis B,"nausea, abdominal pain, jaundice, joint pain"
3,Heart Attack,"chest pain, sweating, arm pain"
4,Common Cold,"runny nose, sore throat, headache"


In [51]:
data.shape

(1037, 2)

In [54]:
data.disease.nunique()

71

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(data['symptoms'])

In [61]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(data['disease'])

In [62]:
from sklearn.model_selection import train_test_split
X_train , X_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [64]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [65]:
from sklearn.metrics import accuracy_score
prd = lr.predict(X_test)
accuracy_score(y_test, prd)

0.8701923076923077

In [66]:
symptom_input = ["fever headache body pain"]

vec = vectorizer.transform(symptom_input)

prediction = lr.predict(vec)

le.inverse_transform(prediction)

array(['Influenza'], dtype=object)

In [67]:
import numpy as np

probs = lr.predict_proba(vec)

top3 = np.argsort(probs[0])[-3:][::-1]

le.inverse_transform(top3)

array(['Influenza', 'Mumps', 'Dengue'], dtype=object)

In [68]:
from sklearn.ensemble import RandomForestClassifier
rf_class = RandomForestClassifier(n_estimators=300, random_state=42)
rf_class.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [70]:
rf_pred = rf_class.predict(X_test)

In [72]:
from sklearn.metrics import accuracy_score
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f'Random forest accuracy: ',rf_accuracy)

Random forest accuracy:  0.8557692307692307


In [73]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf_class, x,y,cv=5)
print(scores.mean())

0.8698439241917504


In [74]:
import numpy as np

symptom_input = ["fever headache body pain"]

vec = vectorizer.transform(symptom_input)

probs = rf_class.predict_proba(vec)

top3 = np.argsort(probs[0])[-3:][::-1]

predicted_diseases = le.inverse_transform(top3)

print(predicted_diseases)

['Mumps' 'Rubella' 'Meningitis']


In [76]:
len(vectorizer.get_feature_names_out())

165

In [77]:
import numpy as np

def predict_top3(symptoms):

    vec = vectorizer.transform([symptoms])

    probs = lr.predict_proba(vec)

    top3_idx = np.argsort(probs[0])[-3:][::-1]

    diseases = le.inverse_transform(top3_idx)

    probabilities = probs[0][top3_idx]

    return list(zip(diseases, probabilities))

In [78]:
predict_top3("fever headache body pain fatigue")

[('Influenza', np.float64(0.17737468806439144)),
 ('Malaria', np.float64(0.09221451982070707)),
 ('Mumps', np.float64(0.08315140059325761))]

In [79]:
predict_top3("high fever chills sweating nausea")

[('Malaria', np.float64(0.5059712758258756)),
 ('Dengue', np.float64(0.0736816939078681)),
 ('Pneumonia', np.float64(0.029584748681648232))]

In [80]:
import numpy as np

def predict_top3(symptoms):

    vec = vectorizer.transform([symptoms])

    probs = lr.predict_proba(vec)

    top3_idx = np.argsort(probs[0])[-3:][::-1]

    diseases = le.inverse_transform(top3_idx)

    probabilities = probs[0][top3_idx]

    results = []

    for d,p in zip(diseases, probabilities):
        results.append((d, round(p*100,2)))

    return results

In [81]:
predict_top3("fever headache body pain fatigue")

[('Influenza', np.float64(17.74)),
 ('Malaria', np.float64(9.22)),
 ('Mumps', np.float64(8.32))]

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=2
)

In [83]:
X = vectorizer.fit_transform(data['symptoms'])

In [84]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(data['disease'])

In [85]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    C=3,
    solver="lbfgs"
)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,3
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [86]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

accuracy_score(y_test, pred)

0.9326923076923077

In [87]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print(scores.mean())

0.9479422147900408


In [90]:
import pickle

pickle.dump(model, open("disease_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))
pickle.dump(le, open("label_encoder.pkl","wb"))

In [91]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=2
)

X = vectorizer.fit_transform(data['symptoms'])

In [92]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(data['disease'])

In [93]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [94]:
import pickle

pickle.dump(model, open("disease_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))
pickle.dump(le, open("label_encoder.pkl","wb"))

NEW START

In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

In [3]:
data = pd.read_csv("curewise_dataset_shuffled.csv")

In [4]:
data = data.dropna()

In [6]:
X = data["symptoms"]  
y = data["disease"]   

In [7]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [8]:
vectorizer = CountVectorizer()
X_vectorized = vectorizer.fit_transform(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y_encoded, test_size=0.2, random_state=42
)

In [10]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [11]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9230769230769231


In [12]:
with open("new_disease_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("new_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("new_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("All files saved successfully")

All files saved successfully
